In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
%%capture
!pip install ultralytics --quiet
!pip install transformers accelerate datasets --quiet
!pip install albumentations --quiet
!pip install segmentation-models-pytorch --quiet
!pip install faiss-cpu --quiet
!pip install einops --quiet

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

import albumentations as A
from albumentations.pytorch import ToTensorV2

from ultralytics import YOLO

print("All imports successful.")

In [ ]:
SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(SEED)
print(f"All random seeds fixed to {SEED}")

In [ ]:
# Cell 5
# ── Dataset Paths ──────────────────────────────────────────────────
BASE_DIR        = Path("/kaggle/input/datasets/solesensei/solesensei_bdd100k")

# Detection (100k images + labels)
DET_IMG_TRAIN   = BASE_DIR / "bdd100k" / "bdd100k" / "images" / "100k" / "train"
DET_IMG_VAL     = BASE_DIR / "bdd100k" / "bdd100k" / "images" / "100k" / "val"
DET_LABEL_DIR   = BASE_DIR / "bdd100k_labels_release" / "bdd100k" / "labels"

# Segmentation (10k images + masks)
SEG_IMG_TRAIN   = BASE_DIR / "bdd100k_seg" / "bdd100k" / "seg" / "images" / "train"
SEG_IMG_VAL     = BASE_DIR / "bdd100k_seg" / "bdd100k" / "seg" / "images" / "val"
SEG_MASK_TRAIN  = BASE_DIR / "bdd100k_seg" / "bdd100k" / "seg" / "labels" / "train"
SEG_MASK_VAL    = BASE_DIR / "bdd100k_seg" / "bdd100k" / "seg" / "labels" / "val"
SEG_COLOR_TRAIN = BASE_DIR / "bdd100k_seg" / "bdd100k" / "seg" / "color_labels" / "train"
SEG_COLOR_VAL   = BASE_DIR / "bdd100k_seg" / "bdd100k" / "seg" / "color_labels" / "val"

# Output directories
OUT_DIR         = Path("/kaggle/working")
YOLO_DIR        = OUT_DIR / "yolo_dataset"
CKPT_DIR        = OUT_DIR / "checkpoints"
RESULTS_DIR     = OUT_DIR / "results"

for d in [YOLO_DIR, CKPT_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── 10 Selected Classes ────────────────────────────────────────────
# Mandatory: car, pedestrian, traffic light, lane marking
# Road hazard proxy: drivable area (BDD100K has no pothole/crack annotations)
# Remaining: chosen by frequency + road safety relevance
SELECTED_CLASSES = [
    "car",            # 0 - mandatory
    "person",         # 1 - mandatory (NOT "pedestrian" — this is the actual name)
    "traffic light",  # 2 - mandatory
    "bike",           # 3 - road hazard proxy / vulnerable user (lane marking is poly2d, not box2d)
    "truck",          # 4 - high frequency, safety critical
    "bus",            # 5 - high frequency, safety critical
    "rider",          # 6 - vulnerable road user
    "motor",          # 7 - vulnerable road user
    "traffic sign",   # 8 - road safety
    "train",          # 9 - vehicle category restriction class
]

CLASS2ID = {cls: idx for idx, cls in enumerate(SELECTED_CLASSES)}
ID2CLASS  = {idx: cls for cls, idx in CLASS2ID.items()}
NUM_CLASSES = len(SELECTED_CLASSES)

# ── YOLO Hyperparameters ───────────────────────────────────────────
YOLO_MODEL    = "yolo11n.pt"
IMG_SIZE      = 640
BATCH_SIZE    = 16
EPOCHS        = 50
LR0           = 0.01
MOMENTUM      = 0.937
PATIENCE      = 10

# ── Segmentation Hyperparameters ──────────────────────────────────
SEG_IMG_SIZE    = 512
SEG_BATCH_SIZE  = 8
SEG_EPOCHS      = 50
SEG_LR          = 6e-5
UNET_LR         = 1e-3

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=== Global Config ===")
print(f"Device        : {DEVICE}")
print(f"Num classes   : {NUM_CLASSES}")
print(f"Classes       : {SELECTED_CLASSES}")
print(f"YOLO model    : {YOLO_MODEL}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Epochs        : {EPOCHS}")
print(f"Output dir    : {OUT_DIR}")

In [ ]:
#Cell 6
paths_to_check = {
    "Det images (train)"  : DET_IMG_TRAIN,
    "Det images (val)"    : DET_IMG_VAL,
    "Det labels dir"      : DET_LABEL_DIR,
    "Seg images (train)"  : SEG_IMG_TRAIN,
    "Seg images (val)"    : SEG_IMG_VAL,
    "Seg masks (train)"   : SEG_MASK_TRAIN,
    "Seg masks (val)"     : SEG_MASK_VAL,
    "Seg color (train)"   : SEG_COLOR_TRAIN,
    "Seg color (val)"     : SEG_COLOR_VAL,
}

all_ok = True
for name, path in paths_to_check.items():
    exists = path.exists()
    status = "✅" if exists else "❌ NOT FOUND"
    print(f"{status}  {name}: {path}")
    if not exists:
        all_ok = False

print()
print("All paths OK ✅" if all_ok else "⚠️  Fix the paths above before proceeding.")

# Also peek inside label dir to find the JSON files
print("\n── Contents of DET_LABEL_DIR ──")
if DET_LABEL_DIR.exists():
    for item in sorted(DET_LABEL_DIR.iterdir()):
        print(f"  {item.name}")
else:
    print("  Directory not found.")

**Phase 1: EDA and Visualization**

In [ ]:
#1.1.1
# ── Find detection label JSONs ─────────────────────────────────────
print("── Contents of DET_LABEL_DIR ──")
for item in sorted(DET_LABEL_DIR.iterdir()):
    print(f"  {item.name}")

# Load train labels
det_label_train_path = DET_LABEL_DIR / "bdd100k_labels_images_train.json"
det_label_val_path   = DET_LABEL_DIR / "bdd100k_labels_images_val.json"

print("\nLoading detection labels...")
with open(det_label_train_path, "r") as f:
    det_train_raw = json.load(f)

with open(det_label_val_path, "r") as f:
    det_val_raw = json.load(f)

print(f"Train samples : {len(det_train_raw):,}")
print(f"Val samples   : {len(det_val_raw):,}")
print(f"\nSample entry keys: {list(det_train_raw[0].keys())}")
print(f"\nSample entry:\n{json.dumps(det_train_raw[0], indent=2)}")

In [ ]:
#1.1.2
def parse_detection_json(raw_data, split_name="train"):
    """
    Parse BDD100K detection JSON into a flat DataFrame.
    Each row = one bounding box annotation.
    """
    records = []
    for entry in tqdm(raw_data, desc=f"Parsing {split_name}"):
        img_name  = entry.get("name", "")
        attrs     = entry.get("attributes", {})
        weather   = attrs.get("weather", "unknown")
        timeofday = attrs.get("timeofday", "unknown")
        scene     = attrs.get("scene", "unknown")
        labels    = entry.get("labels", [])

        if labels is None:
            labels = []

        for label in labels:
            category = label.get("category", "unknown")
            box2d    = label.get("box2d", None)

            if box2d is None:
                continue  # skip non-box labels (e.g. lane polylines)

            x1 = box2d["x1"]
            y1 = box2d["y1"]
            x2 = box2d["x2"]
            y2 = box2d["y2"]
            w  = x2 - x1
            h  = y2 - y1
            area = w * h

            records.append({
                "image"     : img_name,
                "split"     : split_name,
                "category"  : category,
                "weather"   : weather,
                "timeofday" : timeofday,
                "scene"     : scene,
                "x1"        : x1,
                "y1"        : y1,
                "x2"        : x2,
                "y2"        : y2,
                "width"     : w,
                "height"    : h,
                "area"      : area,
            })

    return pd.DataFrame(records)


df_train = parse_detection_json(det_train_raw, "train")
df_val   = parse_detection_json(det_val_raw,   "val")
df_all   = pd.concat([df_train, df_val], ignore_index=True)

print(f"\nTotal annotations : {len(df_all):,}")
print(f"Train annotations : {len(df_train):,}")
print(f"Val annotations   : {len(df_val):,}")
print(f"\nUnique categories ({df_all['category'].nunique()}):")
print(sorted(df_all["category"].unique()))
print(f"\nWeather values    : {sorted(df_all['weather'].unique())}")
print(f"Time of day values: {sorted(df_all['timeofday'].unique())}")
print(f"\nDataFrame shape   : {df_all.shape}")
df_train.head(3)

In [ ]:
#1.1.3
def parse_seg_metadata(img_dir, mask_dir, split_name="train"):
    """
    Build a DataFrame of segmentation image paths + mask paths.
    Mask files are named: <stem>_train_id.png
    Image files are named: <stem>.jpg
    """
    img_dir  = Path(img_dir)
    mask_dir = Path(mask_dir)

    img_files  = sorted(img_dir.glob("*.jpg"))
    # Build lookup: stem -> full mask path (suffix is _train_id.png)
    mask_files = {f.stem.replace("_train_id", ""): f for f in mask_dir.glob("*_train_id.png")}

    records = []
    for img_path in img_files:
        stem      = img_path.stem
        mask_path = mask_files.get(stem, None)
        records.append({
            "image_name" : stem,
            "split"      : split_name,
            "img_path"   : str(img_path),
            "mask_path"  : str(mask_path) if mask_path else None,
            "has_mask"   : mask_path is not None,
        })

    df = pd.DataFrame(records)
    return df


df_seg_train = parse_seg_metadata(SEG_IMG_TRAIN, SEG_MASK_TRAIN, "train")
df_seg_val   = parse_seg_metadata(SEG_IMG_VAL,   SEG_MASK_VAL,   "val")
df_seg_all   = pd.concat([df_seg_train, df_seg_val], ignore_index=True)

print(f"Seg train images    : {len(df_seg_train):,}")
print(f"Seg val images      : {len(df_seg_val):,}")
print(f"Total seg images    : {len(df_seg_all):,}")
print(f"Images with masks   : {df_seg_all['has_mask'].sum():,}")
print(f"Images missing masks: {(~df_seg_all['has_mask']).sum():,}")

# Verify a mask actually loads correctly
sample = df_seg_train[df_seg_train["has_mask"]].iloc[0]
mask   = np.array(Image.open(sample["mask_path"]))
print(f"\nSample mask shape  : {mask.shape}")
print(f"Unique class IDs   : {np.unique(mask)}")
print(f"Sample image path  : {sample['img_path']}")
print(f"Sample mask path   : {sample['mask_path']}")
df_seg_train.head(3)

In [ ]:
#1.1.4
# ── Filter to our 10 selected classes ─────────────────────────────
df_train_10 = df_train[df_train["category"].isin(SELECTED_CLASSES)].copy()
df_val_10   = df_val[df_val["category"].isin(SELECTED_CLASSES)].copy()
df_all_10   = pd.concat([df_train_10, df_val_10], ignore_index=True)

# Add integer class ID
df_train_10["class_id"] = df_train_10["category"].map(CLASS2ID)
df_val_10["class_id"]   = df_val_10["category"].map(CLASS2ID)
df_all_10["class_id"]   = df_all_10["category"].map(CLASS2ID)

print("=== Class Filtering Summary ===")
print(f"Total annotations (all classes) : {len(df_all):,}")
print(f"Total annotations (10 classes)  : {len(df_all_10):,}")
print(f"Retained                        : {len(df_all_10)/len(df_all)*100:.1f}%")
print(f"\nPer-class annotation counts (train):")
print(df_train_10["category"].value_counts().to_string())

# Document filtering decisions
print("\n=== Filtering Decision Log ===")
all_cats = sorted(df_train["category"].unique())
for cat in all_cats:
    count  = (df_train["category"] == cat).sum()
    kept   = "✅ KEPT" if cat in SELECTED_CLASSES else "❌ DROPPED"
    reason = ""
    if cat not in SELECTED_CLASSES:
        reason = "— low frequency / redundant"
    print(f"  {kept}  {cat:<20} ({count:>7,} annotations) {reason}")

In [ ]:
#1.1.5
def show_sample_images(img_dir, n=4, title="Sample Images"):
    img_dir  = Path(img_dir)
    samples  = sorted(img_dir.glob("*.jpg"))[:n]

    fig, axes = plt.subplots(1, n, figsize=(20, 4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for ax, img_path in zip(axes, samples):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=7)
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    print(f"Image size: {img.size} (W x H)")


show_sample_images(DET_IMG_TRAIN,  n=4, title="Detection — 100k Train Samples")
show_sample_images(SEG_IMG_TRAIN,  n=4, title="Segmentation — 10k Train Samples")

In [ ]:
#1.1.6
# BDD100K standard segmentation class mapping (train_id -> class name)
BDD_SEG_CLASSES = {
    0:   "road",
    1:   "sidewalk",
    2:   "building",
    3:   "wall",
    4:   "fence",
    5:   "pole",
    6:   "traffic light",
    7:   "traffic sign",
    8:   "vegetation",
    9:   "terrain",
    10:  "sky",
    11:  "person",
    12:  "rider",
    13:  "car",
    14:  "truck",
    15:  "bus",
    16:  "train",
    17:  "motor",
    18:  "bike",
    255: "unlabeled / ignore",
}

# Check which classes appear in our sample mask
sample_ids = [0, 2, 4, 5, 6, 7, 8, 10, 11, 13, 255]
print("Classes present in sample mask:")
for cid in sample_ids:
    print(f"  ID {cid:>3} → {BDD_SEG_CLASSES.get(cid, 'unknown')}")

# Define our 10 segmentation classes (mapped from BDD train_ids)
# We pick classes that are: (a) present in masks, (b) road-safety relevant
SEG_CLASS_MAP = {
    0:  0,   # road
    11: 1,   # person
    12: 1,   # rider → merged into person (both are vulnerable road users)
    6:  2,   # traffic light
    7:  3,   # traffic sign
    13: 4,   # car
    14: 5,   # truck
    15: 6,   # bus
    16: 7,   # train
    17: 8,   # motor
    18: 9,   # bike
}

SEG_CLASSES = [
    "road", "person", "traffic light", "traffic sign",
    "car", "truck", "bus", "train", "motor", "bike"
]

print(f"\nSegmentation classes ({len(SEG_CLASSES)}):")
for i, name in enumerate(SEG_CLASSES):
    print(f"  {i} → {name}")

print("\nNote: 'lane marking' appears as part of 'road' class in seg masks.")
print("      255 = unlabeled/ignore — excluded from loss computation.")

In [ ]:
#1.2.1
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Category Distribution — BDD100K Detection Annotations", fontsize=14, fontweight="bold")

# All classes
cat_counts_all = df_train["category"].value_counts()
axes[0].bar(cat_counts_all.index, cat_counts_all.values, color="steelblue", edgecolor="black")
axes[0].set_title("All Classes (Train Split)")
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Annotation Count")
axes[0].tick_params(axis="x", rotation=45)
for i, v in enumerate(cat_counts_all.values):
    axes[0].text(i, v + 1000, f"{v:,}", ha="center", fontsize=8)

# Selected 10 classes only
cat_counts_10 = df_train_10["category"].value_counts()
colors = plt.cm.tab10(np.linspace(0, 1, len(cat_counts_10)))
axes[1].bar(cat_counts_10.index, cat_counts_10.values, color=colors, edgecolor="black")
axes[1].set_title("Selected 10 Classes (Train Split)")
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Annotation Count")
axes[1].tick_params(axis="x", rotation=45)
for i, v in enumerate(cat_counts_10.values):
    axes[1].text(i, v + 500, f"{v:,}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_category_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eda_category_distribution.png")

In [ ]:
#1.2.2
print("Sampling images per weather condition for intensity analysis...")

weather_conditions = ["clear", "rainy", "snowy", "foggy", "overcast", "partly cloudy"]
weather_colors     = ["gold", "royalblue", "lightcyan", "lightgray", "slategray", "plum"]
N_SAMPLES          = 100

weather_intensities = {}

for weather in weather_conditions:
    imgs_for_weather = df_train[df_train["weather"] == weather]["image"].unique()
    sampled          = np.random.choice(imgs_for_weather,
                                        size=min(N_SAMPLES, len(imgs_for_weather)),
                                        replace=False)
    intensities = []
    failed      = 0
    for img_name in sampled:
        img_path = DET_IMG_TRAIN / img_name
        if not img_path.exists():
            failed += 1
            continue
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            failed += 1
            continue
        intensities.extend(img.flatten()[::10].tolist())

    weather_intensities[weather] = intensities
    print(f"  {weather:<15}: {len(imgs_for_weather):>5} images | "
          f"sampled {len(sampled)} | failed {failed} | pixels: {len(intensities):,}")

# Plot per-weather histograms
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Pixel Intensity Distribution by Weather Condition", fontsize=14, fontweight="bold")

for ax, weather, color in zip(axes.flatten(), weather_conditions, weather_colors):
    intensities = weather_intensities[weather]
    if len(intensities) == 0:
        ax.text(0.5, 0.5, f"No valid images\nfor '{weather}'",
                ha="center", va="center", transform=ax.transAxes, fontsize=12)
        ax.set_title(f"{weather.capitalize()} (no data)")
        continue
    ax.hist(intensities, bins=64, color=color, edgecolor="black", alpha=0.8, density=True)
    mean_val = np.mean(intensities)
    ax.axvline(mean_val, color="red", linestyle="--",
               linewidth=1.5, label=f"Mean={mean_val:.1f}")
    ax.set_title(f"{weather.capitalize()} (n={len(intensities):,} px)")
    ax.set_xlabel("Pixel Intensity (0–255)")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_weather_intensity.png", dpi=150, bbox_inches="tight")
plt.show()

# Overlay plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title("Overlapping Pixel Intensity Distributions by Weather Condition",
             fontsize=13, fontweight="bold")
for weather, color in zip(weather_conditions, weather_colors):
    intensities = weather_intensities[weather]
    if len(intensities) == 0:
        continue
    ax.hist(intensities, bins=64, color=color, alpha=0.5,
            density=True, label=weather.capitalize(), edgecolor="none")
ax.set_xlabel("Pixel Intensity (0–255)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_weather_intensity_overlay.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eda_weather_intensity.png + eda_weather_intensity_overlay.png")

In [ ]:
#1.2.3
# Time-of-day image counts
tod_image_counts = df_train.groupby("timeofday")["image"].nunique()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Time-of-Day Analysis", fontsize=14, fontweight="bold")

# Bar chart — image count
axes[0].bar(tod_image_counts.index, tod_image_counts.values,
            color=["orange", "navy", "purple", "gray"], edgecolor="black")
axes[0].set_title("Image Count per Time of Day")
axes[0].set_xlabel("Time of Day")
axes[0].set_ylabel("Number of Images")
for i, v in enumerate(tod_image_counts.values):
    axes[0].text(i, v + 50, f"{v:,}", ha="center", fontsize=9)

# Pie chart
axes[1].pie(tod_image_counts.values, labels=tod_image_counts.index,
            autopct="%1.1f%%", colors=["orange", "navy", "purple", "gray"],
            startangle=90)
axes[1].set_title("Time-of-Day Distribution (%)")

# Brightness per time of day
print("Sampling brightness per time-of-day (1-2 min)...")
tod_conditions = ["daytime", "night", "dawn/dusk", "undefined"]
tod_colors     = ["orange", "navy", "purple", "gray"]
N_TOD          = 80

tod_brightness = {}
for tod in tod_conditions:
    imgs = df_train[df_train["timeofday"] == tod]["image"].unique()
    sampled = np.random.choice(imgs, size=min(N_TOD, len(imgs)), replace=False)
    brightness = []
    for img_name in sampled:
        img_path = DET_IMG_TRAIN / img_name
        if img_path.exists():
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                brightness.append(np.mean(img))
    tod_brightness[tod] = brightness

axes[2].boxplot([tod_brightness[t] for t in tod_conditions],
                labels=[t.replace("/", "/\n") for t in tod_conditions],
                patch_artist=True,
                boxprops=dict(facecolor="lightblue"),
                medianprops=dict(color="red", linewidth=2))
axes[2].set_title("Image Brightness by Time of Day")
axes[2].set_xlabel("Time of Day")
axes[2].set_ylabel("Mean Pixel Intensity")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_timeofday_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eda_timeofday_analysis.png")

# Print brightness stats
print("\nBrightness Statistics:")
for tod in tod_conditions:
    b = tod_brightness[tod]
    if b:
        print(f"  {tod:<12}: mean={np.mean(b):.1f}  std={np.std(b):.1f}  "
              f"min={np.min(b):.1f}  max={np.max(b):.1f}")

In [ ]:
#1.2.4
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Class Imbalance Analysis — Selected 10 Classes", fontsize=14, fontweight="bold")

cat_counts = df_train_10["category"].value_counts()
total      = cat_counts.sum()

# Absolute counts with imbalance ratio
colors = ["#2ecc71" if v > total / len(cat_counts) else "#e74c3c"
          for v in cat_counts.values]
bars = axes[0].barh(cat_counts.index, cat_counts.values, color=colors, edgecolor="black")
axes[0].set_title("Annotation Counts (Green=Over-represented, Red=Under-represented)")
axes[0].set_xlabel("Annotation Count")
for bar, v in zip(bars, cat_counts.values):
    axes[0].text(v + 1000, bar.get_y() + bar.get_height() / 2,
                 f"{v:,}", va="center", fontsize=9)
mean_line = total / len(cat_counts)
axes[0].axvline(mean_line, color="black", linestyle="--",
                linewidth=1.5, label=f"Mean = {mean_line:,.0f}")
axes[0].legend()

# Log scale for better visibility
axes[1].barh(cat_counts.index, cat_counts.values, color=colors, edgecolor="black")
axes[1].set_xscale("log")
axes[1].set_title("Annotation Counts (Log Scale)")
axes[1].set_xlabel("Annotation Count (log)")
for bar, v in zip(bars, cat_counts.values):
    axes[1].text(v * 1.05, bar.get_y() + bar.get_height() / 2,
                 f"{v:,}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_class_imbalance.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nClass Imbalance Summary:")
print(f"{'Class':<15} {'Count':>10} {'% of Total':>12} {'Imbalance Ratio':>16}")
print("-" * 55)
for cls, cnt in cat_counts.items():
    pct   = cnt / total * 100
    ratio = cnt / mean_line
    flag  = "▲ OVER" if ratio > 1 else "▼ UNDER"
    print(f"{cls:<15} {cnt:>10,} {pct:>11.1f}% {ratio:>12.2f}x  {flag}")
print("Saved: eda_class_imbalance.png")

In [ ]:
#1.2.5
def draw_bboxes_on_image(img_path, annotations_df, class2color):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    for _, row in annotations_df.iterrows():
        x1, y1, x2, y2 = int(row.x1), int(row.y1), int(row.x2), int(row.y2)
        cat   = row.category
        color = class2color.get(cat, (255, 255, 255))
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = cat
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(img, (x1, y1 - th - 4), (x1 + tw, y1), color, -1)
        cv2.putText(img, label, (x1, y1 - 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
    return img

# Color per class
np.random.seed(SEED)
CLASS_COLORS = {
    cls: tuple(int(c) for c in np.random.randint(50, 230, 3))
    for cls in SELECTED_CLASSES
}

# Pick images that have >= 3 boxes AND the file actually exists
rich_images = (df_train_10.groupby("image")
               .filter(lambda x: len(x) >= 3)["image"].unique())

# Filter to only images that exist on disk
valid_images = [img for img in rich_images
                if (DET_IMG_TRAIN / img).exists()]
print(f"Valid images available: {len(valid_images):,}")

sample_imgs = np.random.choice(valid_images, size=10, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(25, 10))
fig.suptitle("10 Annotated Example Images — BDD100K Detection (Selected 10 Classes)",
             fontsize=14, fontweight="bold")

for ax, img_name in zip(axes.flatten(), sample_imgs):
    img_path   = DET_IMG_TRAIN / img_name
    ann_subset = df_train_10[df_train_10["image"] == img_name]
    annotated  = draw_bboxes_on_image(img_path, ann_subset, CLASS_COLORS)
    if annotated is None:
        ax.text(0.5, 0.5, "Read error", ha="center", va="center",
                transform=ax.transAxes)
        ax.axis("off")
        continue
    ax.imshow(annotated)
    ax.set_title(f"{img_name[:20]}\n({len(ann_subset)} boxes)", fontsize=8)
    ax.axis("off")

handles = [patches.Patch(color=np.array(v) / 255, label=k)
           for k, v in CLASS_COLORS.items()]
fig.legend(handles=handles, loc="lower center", ncol=5,
           fontsize=9, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_annotated_examples.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eda_annotated_examples.png")

In [ ]:
#1.2.6
def overlay_seg_mask(img_path, color_mask_path):
    """Overlay color segmentation mask on image."""
    img        = np.array(Image.open(img_path).convert("RGB"))
    color_mask = np.array(Image.open(color_mask_path).convert("RGB"))
    overlay    = cv2.addWeighted(img, 0.6, color_mask, 0.4, 0)
    return img, color_mask, overlay


# Sample 5 images from segmentation set for overlay visualization
seg_samples = df_seg_train[df_seg_train["has_mask"]].sample(5, random_state=SEED)

# Get corresponding color label paths
color_mask_files = {
    f.stem.replace("_train_color", ""): f
    for f in SEG_COLOR_TRAIN.glob("*_train_color.png")
}

fig, axes = plt.subplots(5, 3, figsize=(18, 22))
fig.suptitle("Segmentation Examples — Image | Color Mask | Overlay",
             fontsize=14, fontweight="bold")

col_titles = ["Original Image", "Segmentation Mask", "Overlay"]
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=12, fontweight="bold")

for i, (_, row) in enumerate(seg_samples.iterrows()):
    img_path        = Path(row["img_path"])
    color_mask_path = color_mask_files.get(row["image_name"], None)

    if color_mask_path is None:
        print(f"  No color mask for {row['image_name']}, skipping.")
        continue

    img, color_mask, overlay = overlay_seg_mask(img_path, color_mask_path)

    axes[i][0].imshow(img)
    axes[i][0].axis("off")
    axes[i][1].imshow(color_mask)
    axes[i][1].axis("off")
    axes[i][2].imshow(overlay)
    axes[i][2].axis("off")
    axes[i][0].set_ylabel(row["image_name"][:15], fontsize=8, rotation=0,
                          labelpad=60, va="center")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_segmentation_overlays.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eda_segmentation_overlays.png")

In [ ]:
#1.2.7
justification_data = {
    "Class"        : SELECTED_CLASSES,
    "Train Count"  : [df_train_10[df_train_10["category"] == c].shape[0]
                      for c in SELECTED_CLASSES],
    "% of Total"   : [round(df_train_10[df_train_10["category"] == c].shape[0]
                      / len(df_train_10) * 100, 2) for c in SELECTED_CLASSES],
    "Mandatory"    : ["✅" if c in ["car", "person", "traffic light"] else
                      "⚠️ proxy" if c == "bike" else "—"
                      for c in SELECTED_CLASSES],
    "Safety Role"  : [
        "Primary road agent",
        "Vulnerable road user (mandatory)",
        "Intersection control (mandatory)",
        "Road hazard proxy — vulnerable 2-wheeler",
        "Heavy vehicle — collision risk",
        "Heavy vehicle — collision risk",
        "Mixed traffic — high risk",
        "Motorized 2-wheeler — high risk",
        "Speed/regulation enforcement",
        "Rail vehicle — restricted zones",
    ],
}

df_justification = pd.DataFrame(justification_data)
print("=== 10-Class Subset Justification ===\n")
print(df_justification.to_string(index=False))

print("""
=== Filtering Decisions ===
DROPPED 'lane'         : Annotated as poly2d polylines — no bounding boxes available.
                         Lane markings ARE covered in the segmentation task (road class).
DROPPED 'drivable area': Annotated as poly2d polygons — incompatible with YOLO bbox format.
DROPPED 'train'        : Only 136 annotations — statistically insufficient for training.
NOTE    'pedestrian'   : BDD100K uses 'person' as the actual category name.
NOTE    'bicycle'      : BDD100K uses 'bike' as the actual category name.
NOTE    'pothole/crack': Not annotated in BDD100K. Road surface hazards are represented
                         via the segmentation pipeline's road class boundary analysis.
""")

df_justification.to_csv(RESULTS_DIR / "eda_class_justification.csv", index=False)
print("Saved: eda_class_justification.csv")

In [ ]:
#1.2.8
eda_summary = {
    "total_train_images"          : df_train["image"].nunique(),
    "total_val_images"            : df_val["image"].nunique(),
    "total_train_annotations"     : len(df_train),
    "total_10class_annotations"   : len(df_train_10),
    "seg_train_images"            : len(df_seg_train),
    "seg_val_images"              : len(df_seg_val),
    "num_weather_conditions"      : df_train["weather"].nunique(),
    "num_timeofday_conditions"    : df_train["timeofday"].nunique(),
    "most_common_class"           : df_train_10["category"].value_counts().idxmax(),
    "least_common_class"          : df_train_10["category"].value_counts().idxmin(),
    "imbalance_ratio"             : round(
                                        df_train_10["category"].value_counts().max() /
                                        df_train_10["category"].value_counts().min(), 1),
}

df_eda_summary = pd.DataFrame([eda_summary]).T.rename(columns={0: "value"})
print("=== EDA Summary ===")
print(df_eda_summary.to_string())
df_eda_summary.to_csv(RESULTS_DIR / "eda_summary.csv")
print("\nSaved: eda_summary.csv")

# **Phase 3 — Semantic Segmentation** 

In [ ]:
#3.1.1
# ── Segmentation Subset Sizes ──────────────────────────────────────
N_SEG_TRAIN = 4000   # out of 7000
N_SEG_VAL   = 500    # out of 1000
N_SEG_TEST  = 200    # remaining val images used as test

# ── Segmentation Class Mapping ─────────────────────────────────────
SEG_CLASS_MAP = {
    0:  0,   # road
    11: 1,   # person
    12: 1,   # rider → merged into person
    6:  2,   # traffic light
    7:  3,   # traffic sign
    13: 4,   # car
    14: 5,   # truck
    15: 6,   # bus
    16: 7,   # train
    17: 8,   # motor
    18: 9,   # bike
}
IGNORE_INDEX = 255  # all unlabeled pixels

SEG_CLASSES = [
    "road", "person", "traffic light", "traffic sign",
    "car", "truck", "bus", "train", "motor", "bike"
]
NUM_SEG_CLASSES = len(SEG_CLASSES)

print(f"Segmentation classes : {NUM_SEG_CLASSES}")
print(f"Train subset         : {N_SEG_TRAIN}")
print(f"Val subset           : {N_SEG_VAL}")
print(f"Test subset          : {N_SEG_TEST}")
print(f"Device               : {DEVICE}")

In [ ]:
#3.1.2
# Shuffle and subset the seg dataframes
np.random.seed(SEED)

# Train subset
train_indices = np.random.choice(len(df_seg_train), size=N_SEG_TRAIN, replace=False)
df_seg_train_subset = df_seg_train.iloc[train_indices].reset_index(drop=True)

# Val/Test subset from val split
val_indices  = np.random.permutation(len(df_seg_val))
df_seg_val_subset  = df_seg_val.iloc[val_indices[:N_SEG_VAL]].reset_index(drop=True)
df_seg_test_subset = df_seg_val.iloc[val_indices[N_SEG_VAL:N_SEG_VAL+N_SEG_TEST]].reset_index(drop=True)

print(f"Train subset size : {len(df_seg_train_subset):,}")
print(f"Val subset size   : {len(df_seg_val_subset):,}")
print(f"Test subset size  : {len(df_seg_test_subset):,}")
print(f"\nSample train entry:")
print(df_seg_train_subset.iloc[0])

In [ ]:
#3.1.3
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Training Augmentation ──────────────────────────────────────────
train_transform = A.Compose([
    A.RandomResizedCrop(
        size=(SEG_IMG_SIZE, SEG_IMG_SIZE),
        scale=(0.5, 1.0),       # FIXED: must be between 0 and 1
        ratio=(0.75, 1.33),
        p=1.0
    ),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.ColorJitter(
        brightness=0.4,
        contrast=0.4,
        saturation=0.4,
        hue=0.1,
        p=0.8
    ),
    A.HueSaturationValue(
        hue_shift_limit=20,
        sat_shift_limit=30,
        val_shift_limit=20,
        p=0.5
    ),
    A.ShiftScaleRotate(
        shift_limit=0.1,
        scale_limit=0.2,
        rotate_limit=10,
        border_mode=0,
        p=0.5
    ),
    A.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    ),
    ToTensorV2(),
])

# ── Validation/Test Transform ──────────────────────────────────────
val_transform = A.Compose([
    A.Resize(SEG_IMG_SIZE, SEG_IMG_SIZE),
    A.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    ),
    ToTensorV2(),
])

print("Augmentation pipelines defined.")
print(f"Train transforms : {len(train_transform.transforms)} steps")
print(f"Val transforms   : {len(val_transform.transforms)} steps")

In [ ]:
#3.1.4
class BDD100KSegDataset(Dataset):
    def __init__(self, df, seg_class_map, ignore_index=255, transform=None):
        self.df            = df.reset_index(drop=True)
        self.seg_class_map = seg_class_map
        self.ignore_index  = ignore_index
        self.transform     = transform

    def __len__(self):
        return len(self.df)

    def remap_mask(self, mask_np):
        """Remap BDD100K train_ids to our 0–9 class indices."""
        remapped = np.full_like(mask_np, self.ignore_index, dtype=np.int64)
        for src_id, dst_id in self.seg_class_map.items():
            remapped[mask_np == src_id] = dst_id
        return remapped

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = row["img_path"]
        msk_path = row["mask_path"]

        # Load image
        image = np.array(Image.open(img_path).convert("RGB"))

        # Load and remap mask
        mask_raw = np.array(Image.open(msk_path))
        mask     = self.remap_mask(mask_raw)

        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]           # (C, H, W) tensor
            mask  = transformed["mask"].long()     # (H, W) tensor

        return image, mask


# Quick test
test_ds  = BDD100KSegDataset(df_seg_train_subset, SEG_CLASS_MAP,
                              transform=val_transform)
img, msk = test_ds[0]
print(f"Image tensor shape : {img.shape}")
print(f"Mask tensor shape  : {msk.shape}")
print(f"Unique mask values : {msk.unique().tolist()}")
print(f"Dataset size       : {len(test_ds)}")

In [ ]:
#3.1.5
# ── Datasets ──────────────────────────────────────────────────────
train_dataset = BDD100KSegDataset(
    df_seg_train_subset, SEG_CLASS_MAP,
    transform=train_transform
)
val_dataset = BDD100KSegDataset(
    df_seg_val_subset, SEG_CLASS_MAP,
    transform=val_transform
)
test_dataset = BDD100KSegDataset(
    df_seg_test_subset, SEG_CLASS_MAP,
    transform=val_transform
)

# ── DataLoaders ────────────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset,
    batch_size  = SEG_BATCH_SIZE,
    shuffle     = True,
    num_workers = 2,
    pin_memory  = True,
    drop_last   = True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size  = SEG_BATCH_SIZE,
    shuffle     = False,
    num_workers = 2,
    pin_memory  = True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size  = SEG_BATCH_SIZE,
    shuffle     = False,
    num_workers = 2,
    pin_memory  = True,
)

print("=== DataLoader Summary ===")
print(f"Train batches : {len(train_loader):,}  ({len(train_dataset):,} images)")
print(f"Val batches   : {len(val_loader):,}  ({len(val_dataset):,} images)")
print(f"Test batches  : {len(test_loader):,}  ({len(test_dataset):,} images)")

# Verify one batch
imgs, masks = next(iter(train_loader))
print(f"\nBatch image shape : {imgs.shape}")
print(f"Batch mask shape  : {masks.shape}")
print(f"Mask unique vals  : {masks.unique().tolist()}")

In [ ]:
#3.1.6
def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Reverse ImageNet normalization for visualization."""
    t = tensor.clone()
    for i, (m, s) in enumerate(zip(mean, std)):
        t[i] = t[i] * s + m
    return t.clamp(0, 1)

def mask_to_color(mask_np, n_classes=NUM_SEG_CLASSES):
    """Convert class index mask to RGB color image."""
    colors = plt.cm.tab10(np.linspace(0, 1, n_classes))[:, :3]
    h, w   = mask_np.shape
    rgb    = np.zeros((h, w, 3))
    for cls_id in range(n_classes):
        rgb[mask_np == cls_id] = colors[cls_id]
    rgb[mask_np == 255] = [0.2, 0.2, 0.2]  # ignore = dark gray
    return rgb


# Visualize 4 augmented training samples
fig, axes = plt.subplots(4, 2, figsize=(12, 16))
fig.suptitle("Augmented Training Samples — Image & Remapped Mask",
             fontsize=13, fontweight="bold")

aug_ds = BDD100KSegDataset(df_seg_train_subset, SEG_CLASS_MAP,
                            transform=train_transform)

for i in range(4):
    img_t, msk_t = aug_ds[i]
    img_vis = denormalize(img_t).permute(1, 2, 0).numpy()
    msk_vis = mask_to_color(msk_t.numpy())

    axes[i][0].imshow(img_vis)
    axes[i][0].set_title(f"Image {i+1}", fontsize=9)
    axes[i][0].axis("off")

    axes[i][1].imshow(msk_vis)
    axes[i][1].set_title(f"Mask {i+1}", fontsize=9)
    axes[i][1].axis("off")

# Add class legend
handles = [patches.Patch(color=plt.cm.tab10(i / NUM_SEG_CLASSES), label=SEG_CLASSES[i])
           for i in range(NUM_SEG_CLASSES)]
fig.legend(handles=handles, loc="lower center", ncol=5,
           fontsize=8, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig(RESULTS_DIR / "seg_augmented_samples.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: seg_augmented_samples.png")

In [ ]:
#3.1.7
class CombinedLoss(nn.Module):
    """
    Combined CrossEntropy + Dice Loss (equal weight).
    Used for both SegFormer and U-Net as required by project.
    """
    def __init__(self, num_classes=NUM_SEG_CLASSES, ignore_index=255, weight=0.5):
        super().__init__()
        self.weight       = weight
        self.ignore_index = ignore_index
        self.num_classes  = num_classes
        self.ce_loss      = nn.CrossEntropyLoss(ignore_index=ignore_index)

    def dice_loss(self, preds, targets):
        smooth = 1e-6
        # preds: (B, C, H, W) logits → softmax
        preds  = torch.softmax(preds, dim=1)
        dice   = 0.0
        count  = 0
        for cls in range(self.num_classes):
            pred_c   = preds[:, cls]
            target_c = (targets == cls).float()
            valid    = (targets != self.ignore_index).float()
            pred_c   = pred_c * valid
            target_c = target_c * valid
            inter    = (pred_c * target_c).sum()
            union    = pred_c.sum() + target_c.sum()
            if union > 0:
                dice  += (2.0 * inter + smooth) / (union + smooth)
                count += 1
        return 1.0 - (dice / max(count, 1))

    def forward(self, preds, targets):
        ce   = self.ce_loss(preds, targets)
        dice = self.dice_loss(preds, targets)
        return self.weight * ce + self.weight * dice, ce, dice


criterion = CombinedLoss(
    num_classes  = NUM_SEG_CLASSES,
    ignore_index = IGNORE_INDEX,
    weight       = 0.5
)
print("Combined CE + Dice loss defined.")

# Quick test
dummy_pred   = torch.randn(2, NUM_SEG_CLASSES, 64, 64)
dummy_target = torch.randint(0, NUM_SEG_CLASSES, (2, 64, 64))
total, ce, dice = criterion(dummy_pred, dummy_target)
print(f"Test loss — Total: {total:.4f}  CE: {ce:.4f}  Dice: {dice:.4f}")

In [ ]:
#3.1.8
def compute_miou(preds, targets, num_classes=NUM_SEG_CLASSES, ignore_index=255):
    """
    Compute mean IoU across all classes.
    preds   : (B, C, H, W) logits
    targets : (B, H, W) class indices
    """
    pred_labels = preds.argmax(dim=1)  # (B, H, W)
    ious        = []

    for cls in range(num_classes):
        pred_mask   = (pred_labels == cls)
        target_mask = (targets == cls)
        valid_mask  = (targets != ignore_index)

        pred_mask   = pred_mask & valid_mask
        target_mask = target_mask & valid_mask

        intersection = (pred_mask & target_mask).sum().item()
        union        = (pred_mask | target_mask).sum().item()

        if union == 0:
            continue  # class not present — skip
        ious.append(intersection / union)

    return np.mean(ious) if ious else 0.0


def compute_dice(preds, targets, num_classes=NUM_SEG_CLASSES, ignore_index=255):
    """Compute mean Dice score across all classes."""
    pred_labels = preds.argmax(dim=1)
    dices       = []

    for cls in range(num_classes):
        pred_mask   = (pred_labels == cls)
        target_mask = (targets == cls)
        valid_mask  = (targets != ignore_index)

        pred_mask   = pred_mask & valid_mask
        target_mask = target_mask & valid_mask

        tp    = (pred_mask & target_mask).sum().item()
        fp_fn = pred_mask.sum().item() + target_mask.sum().item()

        if fp_fn == 0:
            continue
        dices.append(2 * tp / fp_fn)

    return np.mean(dices) if dices else 0.0


print("mIoU and Dice metric functions defined.")

# Quick test
dummy_pred   = torch.randn(2, NUM_SEG_CLASSES, 64, 64)
dummy_target = torch.randint(0, NUM_SEG_CLASSES, (2, 64, 64))
print(f"Test mIoU : {compute_miou(dummy_pred, dummy_target):.4f}")
print(f"Test Dice : {compute_dice(dummy_pred, dummy_target):.4f}")

# **Model A: SegFormer** 

In [ ]:
#3.2.1
from transformers import SegformerForSemanticSegmentation
import transformers

print(f"Transformers version: {transformers.__version__}")

# Load SegFormer-b0 with random segmentation head (no BDD100K weights)
segformer = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512",
    num_labels        = NUM_SEG_CLASSES,
    id2label          = {i: cls for i, cls in enumerate(SEG_CLASSES)},
    label2id          = {cls: i for i, cls in enumerate(SEG_CLASSES)},
    ignore_mismatched_sizes = True,   # replaces head with new random head
)

# Wrap with DataParallel for T4 x2
segformer = segformer.to(DEVICE)

segformer = segformer.to(DEVICE)

# Count parameters
total_params     = sum(p.numel() for p in segformer.parameters())
trainable_params = sum(p.numel() for p in segformer.parameters() if p.requires_grad)
print(f"\nTotal parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Model on device      : {DEVICE}")

In [ ]:
#3.2.2
from torch.optim.lr_scheduler import CosineAnnealingLR

segformer_optimizer = torch.optim.AdamW(
    segformer.parameters(),
    lr           = SEG_LR,        # 6e-5
    weight_decay = 0.01,
)

segformer_scheduler = CosineAnnealingLR(
    segformer_optimizer,
    T_max = SEG_EPOCHS,
    eta_min = 1e-6,
)

from torch.cuda.amp import GradScaler
seg_scaler = GradScaler()
print("AMP GradScaler initialized.")

print(f"Optimizer  : AdamW  lr={SEG_LR}  weight_decay=0.01")
print(f"Scheduler  : CosineAnnealingLR  T_max={SEG_EPOCHS}  eta_min=1e-6")

In [ ]:
#3.2.3
import time

from torch.cuda.amp import autocast

def train_segformer_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_ce   = 0.0
    total_dice = 0.0

    for imgs, masks in tqdm(loader, desc="Train", leave=False):
        imgs  = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        with autocast():                          # ✅ mixed precision
            outputs = model(pixel_values=imgs)
            logits  = outputs.logits
            logits_up = torch.nn.functional.interpolate(
                logits, size=masks.shape[-2:],
                mode="bilinear", align_corners=False,
            )
            loss, ce, dice = criterion(logits_up, masks)

        seg_scaler.scale(loss).backward()         # ✅ scaled backward
        seg_scaler.step(optimizer)                # ✅ scaled step
        seg_scaler.update()                       # ✅ update scaler

        total_loss += loss.item()
        total_ce   += ce.item()
        total_dice += dice.item()

    n = len(loader)
    return total_loss / n, total_ce / n, total_dice / n


def val_segformer_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_miou = 0.0
    total_dice = 0.0

    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc="Val", leave=False):
            imgs  = imgs.to(device)
            masks = masks.to(device)

            with autocast():                      # ✅ mixed precision
                outputs   = model(pixel_values=imgs)
                logits    = outputs.logits
                del outputs                       # ✅ free HF wrapper immediately
                logits_up = torch.nn.functional.interpolate(
                    logits, size=masks.shape[-2:],
                    mode="bilinear", align_corners=False,
                )
                loss, _, _ = criterion(logits_up, masks)

            miou = compute_miou(logits_up.cpu(), masks.cpu())
            dice = compute_dice(logits_up.cpu(), masks.cpu())

            total_loss += loss.item()
            total_miou += miou
            total_dice += dice

    n = len(loader)
    return total_loss / n, total_miou / n, total_dice / n


# ── Training ───────────────────────────────────────────────────────
print("Starting SegFormer training...")
print(f"Epochs: {SEG_EPOCHS} | Batch: {SEG_BATCH_SIZE} | LR: {SEG_LR}")
print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

seg_history = {
    "train_loss": [], "train_ce": [], "train_dice": [],
    "val_loss":   [], "val_miou": [], "val_dice":   [],
    "lr":         [],
}

best_val_miou    = 0.0
best_epoch       = 0
patience_counter = 0
EARLY_STOP       = 10
seg_train_start  = time.time()

for epoch in range(1, SEG_EPOCHS + 1):
    ep_start = time.time()

    train_loss, train_ce, train_dice = train_segformer_epoch(
        segformer, train_loader, segformer_optimizer, criterion, DEVICE)

    val_loss, val_miou, val_dice = val_segformer_epoch(
        segformer, val_loader, criterion, DEVICE)

    segformer_scheduler.step()
    current_lr = segformer_scheduler.get_last_lr()[0]

    # Log history
    seg_history["train_loss"].append(train_loss)
    seg_history["train_ce"].append(train_ce)
    seg_history["train_dice"].append(train_dice)
    seg_history["val_loss"].append(val_loss)
    seg_history["val_miou"].append(val_miou)
    seg_history["val_dice"].append(val_dice)
    seg_history["lr"].append(current_lr)

    ep_time = time.time() - ep_start
    print(f"Epoch [{epoch:02d}/{SEG_EPOCHS}] "
          f"TrainLoss={train_loss:.4f} "
          f"ValLoss={val_loss:.4f} "
          f"ValmIoU={val_miou:.4f} "
          f"ValDice={val_dice:.4f} "
          f"LR={current_lr:.2e} "
          f"Time={ep_time:.1f}s")

    # Save best model
    if val_miou > best_val_miou:
        best_val_miou    = val_miou
        best_epoch       = epoch
        patience_counter = 0
        # Unwrap DataParallel if needed
        model_to_save = segformer.module if hasattr(segformer, "module") else segformer
        model_to_save.save_pretrained(str(CKPT_DIR / "segformer_best"))
        torch.save(model_to_save.state_dict(),
                   str(CKPT_DIR / "segformer_best.pt"))
        print(f"  ✅ Best model saved (mIoU={best_val_miou:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP:
            print(f"\nEarly stopping at epoch {epoch}. "
                  f"Best mIoU={best_val_miou:.4f} at epoch {best_epoch}")
            break

    torch.cuda.empty_cache()

seg_train_time = (time.time() - seg_train_start) / 60
print(f"\nSegFormer training complete in {seg_train_time:.1f} minutes")
print(f"Best Val mIoU : {best_val_miou:.4f} at epoch {best_epoch}")

In [ ]:
#3.2.4
epochs_ran = len(seg_history["train_loss"])
ep_range   = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("SegFormer Training Curves", fontsize=14, fontweight="bold")

# Loss curves
axes[0].plot(ep_range, seg_history["train_loss"],
             label="Train Loss", color="royalblue", linewidth=2)
axes[0].plot(ep_range, seg_history["val_loss"],
             label="Val Loss",   color="tomato",    linewidth=2)
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# mIoU curve
axes[1].plot(ep_range, seg_history["val_miou"],
             color="green", linewidth=2)
axes[1].set_title("Validation mIoU")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("mIoU")
axes[1].grid(True, alpha=0.3)

# Dice curve
axes[2].plot(ep_range, seg_history["val_dice"],
             color="purple", linewidth=2)
axes[2].set_title("Validation Dice Score")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Dice")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "segformer_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: segformer_training_curves.png")

# Save training history
pd.DataFrame(seg_history).to_csv(
    RESULTS_DIR / "segformer_training_history.csv", index=False)
print("Saved: segformer_training_history.csv")

In [ ]:
#3.2.5
import time

# Load best model
model_to_eval = SegformerForSemanticSegmentation.from_pretrained(
    str(CKPT_DIR / "segformer_best"),
    num_labels              = NUM_SEG_CLASSES,
    ignore_mismatched_sizes = True,
)
model_to_eval = model_to_eval.to(DEVICE)
model_to_eval.eval()

# Per-class IoU and Dice
per_class_iou  = np.zeros(NUM_SEG_CLASSES)
per_class_dice = np.zeros(NUM_SEG_CLASSES)
class_counts   = np.zeros(NUM_SEG_CLASSES)

total_miou = 0.0
total_dice = 0.0
n_batches  = 0

seg_infer_times = []

with torch.no_grad():
    for imgs, masks in tqdm(test_loader, desc="Evaluating SegFormer"):
        imgs  = imgs.to(DEVICE)
        masks = masks.to(DEVICE)

        t0      = time.time()
        outputs = model_to_eval(pixel_values=imgs)
        logits  = outputs.logits
        logits_up = torch.nn.functional.interpolate(
            logits,
            size=(masks.shape[-2:]),
            mode="bilinear",
            align_corners=False,
        )
        t1 = time.time()
        seg_infer_times.append((t1 - t0) / imgs.shape[0] * 1000)

        pred_labels = logits_up.argmax(dim=1).cpu()
        masks_cpu   = masks.cpu()

        # Per-class IoU
        for cls in range(NUM_SEG_CLASSES):
            pred_c   = (pred_labels == cls)
            target_c = (masks_cpu   == cls)
            valid    = (masks_cpu   != IGNORE_INDEX)
            pred_c   = pred_c & valid
            target_c = target_c & valid
            inter    = (pred_c & target_c).sum().item()
            union    = (pred_c | target_c).sum().item()
            if union > 0:
                per_class_iou[cls]  += inter / union
                class_counts[cls]   += 1
            tp    = inter
            fp_fn = pred_c.sum().item() + target_c.sum().item()
            if fp_fn > 0:
                per_class_dice[cls] += 2 * tp / fp_fn

        total_miou += compute_miou(logits_up.cpu(), masks_cpu)
        total_dice += compute_dice(logits_up.cpu(), masks_cpu)
        n_batches  += 1

# Normalize
per_class_iou  = per_class_iou  / np.maximum(class_counts, 1)
per_class_dice = per_class_dice / np.maximum(class_counts, 1)
mean_miou      = total_miou / n_batches
mean_dice      = total_dice / n_batches
avg_infer_ms   = np.mean(seg_infer_times)

print("=== SegFormer Test Set Evaluation ===")
print(f"Mean mIoU        : {mean_miou:.4f}")
print(f"Mean Dice        : {mean_dice:.4f}")
print(f"Inference speed  : {avg_infer_ms:.2f} ms/image")
print(f"\nPer-class IoU:")
for i, cls in enumerate(SEG_CLASSES):
    print(f"  {cls:<15}: IoU={per_class_iou[i]:.4f}  Dice={per_class_dice[i]:.4f}")

# Save results
df_segformer_results = pd.DataFrame({
    "class"    : SEG_CLASSES + ["MEAN"],
    "iou"      : list(per_class_iou) + [mean_miou],
    "dice"     : list(per_class_dice) + [mean_dice],
    "model"    : "segformer",
})
df_segformer_results.to_csv(RESULTS_DIR / "segformer_results.csv", index=False)
print("\nSaved: segformer_results.csv")

# Save timing info
seg_timing = {
    "model"              : "segformer",
    "train_time_minutes" : round(seg_train_time, 2),
    "inference_ms"       : round(avg_infer_ms, 2),
    "mean_miou"          : round(mean_miou, 4),
    "mean_dice"          : round(mean_dice, 4),
}
pd.DataFrame([seg_timing]).to_csv(RESULTS_DIR / "segformer_timing.csv", index=False)
print("Saved: segformer_timing.csv")

In [ ]:
#3.2.6
def visualize_segformer_predictions(model, dataset, n=8, save_path=None):
    model.eval()
    indices = np.random.choice(len(dataset), size=n, replace=False)

    fig, axes = plt.subplots(n, 3, figsize=(15, n * 4))
    fig.suptitle("SegFormer Predictions — Input | Ground Truth | Prediction",
                 fontsize=13, fontweight="bold")

    col_titles = ["Input Image", "Ground Truth", "SegFormer Prediction"]
    for ax, title in zip(axes[0], col_titles):
        ax.set_title(title, fontsize=11, fontweight="bold")

    with torch.no_grad():
        for row, idx in enumerate(indices):
            img_t, msk_t = dataset[idx]
            img_input    = img_t.unsqueeze(0).to(DEVICE)

            outputs   = model(pixel_values=img_input)
            logits    = outputs.logits
            logits_up = torch.nn.functional.interpolate(
                logits,
                size          = msk_t.shape[-2:],
                mode          = "bilinear",
                align_corners = False,
            )
            pred = logits_up.argmax(dim=1).squeeze(0).cpu().numpy()

            img_vis = denormalize(img_t).permute(1, 2, 0).numpy()
            gt_vis  = mask_to_color(msk_t.numpy())
            pr_vis  = mask_to_color(pred)

            axes[row][0].imshow(img_vis)
            axes[row][0].axis("off")
            axes[row][1].imshow(gt_vis)
            axes[row][1].axis("off")
            axes[row][2].imshow(pr_vis)
            axes[row][2].axis("off")

    # Legend
    handles = [patches.Patch(color=plt.cm.tab10(i / NUM_SEG_CLASSES),
                              label=SEG_CLASSES[i])
               for i in range(NUM_SEG_CLASSES)]
    fig.legend(handles=handles, loc="lower center", ncol=5,
               fontsize=8, bbox_to_anchor=(0.5, -0.01))

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


visualize_segformer_predictions(
    model_to_eval,
    test_dataset,
    n         = 8,
    save_path = RESULTS_DIR / "segformer_qualitative.png"
)

In [ ]:
#3.2.7
if torch.cuda.is_available():
    print("=== GPU Memory Usage (SegFormer) ===")
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1e6
        reserved  = torch.cuda.memory_reserved(i) / 1e6
        total     = torch.cuda.get_device_properties(i).total_memory / 1e6
        print(f"  GPU {i}: Allocated={allocated:.1f}MB  "
              f"Reserved={reserved:.1f}MB  Total={total:.1f}MB")
else:
    print("No GPU available.")

# **Model B: U-Net from Scratch**

In [ ]:
#3.3.1
class DoubleConv(nn.Module):
    """Two consecutive Conv2d → BN → ReLU blocks."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class EncoderBlock(nn.Module):
    """DoubleConv + MaxPool downsampling."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        skip = self.conv(x)
        down = self.pool(skip)
        return skip, down


class DecoderBlock(nn.Module):
    """Upsample + skip connection + DoubleConv."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_channels, in_channels // 2,
                                        kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        # Handle size mismatch due to odd dimensions
        if x.shape != skip.shape:
            x = torch.nn.functional.interpolate(
                x, size=skip.shape[2:],
                mode="bilinear", align_corners=False)
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


class UNet(nn.Module):
    """
    Standard 4-level U-Net from scratch.
    Encoder: 4-level conv downsampling with max-pooling
    Decoder: 4-level upsampling with skip connections
    """
    def __init__(self, in_channels=3, num_classes=NUM_SEG_CLASSES,
                 features=[64, 128, 256, 512]):
        super().__init__()

        # ── Encoder ───────────────────────────────────────────────
        self.enc1 = EncoderBlock(in_channels, features[0])
        self.enc2 = EncoderBlock(features[0], features[1])
        self.enc3 = EncoderBlock(features[1], features[2])
        self.enc4 = EncoderBlock(features[2], features[3])

        # ── Bottleneck ────────────────────────────────────────────
        self.bottleneck = DoubleConv(features[3], features[3] * 2)

        # ── Decoder ───────────────────────────────────────────────
        self.dec4 = DecoderBlock(features[3] * 2, features[3])
        self.dec3 = DecoderBlock(features[3],     features[2])
        self.dec2 = DecoderBlock(features[2],     features[1])
        self.dec1 = DecoderBlock(features[1],     features[0])

        # ── Output head ───────────────────────────────────────────
        self.output_conv = nn.Conv2d(features[0], num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        skip1, x = self.enc1(x)
        skip2, x = self.enc2(x)
        skip3, x = self.enc3(x)
        skip4, x = self.enc4(x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decoder
        x = self.dec4(x, skip4)
        x = self.dec3(x, skip3)
        x = self.dec2(x, skip2)
        x = self.dec1(x, skip1)

        return self.output_conv(x)


# Instantiate model
unet = UNet(in_channels=3, num_classes=NUM_SEG_CLASSES)

# Wrap with DataParallel for T4 x2
unet = unet.to(DEVICE)

unet = unet.to(DEVICE)

# Count parameters
total_params     = sum(p.numel() for p in unet.parameters())
trainable_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Model on device      : {DEVICE}")

# Quick forward pass test
dummy_input  = torch.randn(2, 3, SEG_IMG_SIZE, SEG_IMG_SIZE).to(DEVICE)
dummy_output = unet(dummy_input)
print(f"\nInput shape  : {dummy_input.shape}")
print(f"Output shape : {dummy_output.shape}")

In [ ]:
#3.3.2
unet_optimizer = torch.optim.Adam(
    unet.parameters(),
    lr           = UNET_LR,     # 1e-3
    weight_decay = 1e-4,
)

unet_scheduler = CosineAnnealingLR(
    unet_optimizer,
    T_max   = SEG_EPOCHS,
    eta_min = 1e-6,
)

unet_scaler = GradScaler()
print("AMP GradScaler initialized for U-Net.")

print(f"Optimizer  : Adam  lr={UNET_LR}  weight_decay=1e-4")
print(f"Scheduler  : CosineAnnealingLR  T_max={SEG_EPOCHS}  eta_min=1e-6")

In [ ]:
#3.3.3
def train_unet_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_ce   = 0.0
    total_dice = 0.0

    for imgs, masks in tqdm(loader, desc="Train", leave=False):
        imgs  = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        with autocast():                          # ✅
            logits         = model(imgs)
            loss, ce, dice = criterion(logits, masks)

        unet_scaler.scale(loss).backward()        # ✅
        unet_scaler.step(optimizer)               # ✅
        unet_scaler.update()                      # ✅

        total_loss += loss.item()
        total_ce   += ce.item()
        total_dice += dice.item()

    n = len(loader)
    return total_loss / n, total_ce / n, total_dice / n


with torch.no_grad():
        for imgs, masks in tqdm(loader, desc="Val", leave=False):
            imgs  = imgs.to(device)
            masks = masks.to(device)

            with autocast():                      # ✅ add this
                logits     = model(imgs)
                loss, _, _ = criterion(logits, masks)

            miou = compute_miou(logits.cpu(), masks.cpu())
            dice = compute_dice(logits.cpu(), masks.cpu())
            total_loss += loss.item()
            total_miou += miou
            total_dice += dice


# ── Training ───────────────────────────────────────────────────────
print("Starting U-Net training...")
print(f"Epochs: {SEG_EPOCHS} | Batch: {SEG_BATCH_SIZE} | LR: {UNET_LR}")
print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

unet_history = {
    "train_loss": [], "train_ce": [], "train_dice": [],
    "val_loss":   [], "val_miou": [], "val_dice":   [],
    "lr":         [],
}

best_unet_miou   = 0.0
best_unet_epoch  = 0
unet_patience    = 0
EARLY_STOP       = 10
unet_train_start = time.time()

for epoch in range(1, SEG_EPOCHS + 1):
    ep_start = time.time()

    train_loss, train_ce, train_dice = train_unet_epoch(
        unet, train_loader, unet_optimizer, criterion, DEVICE)

    val_loss, val_miou, val_dice = val_unet_epoch(
        unet, val_loader, criterion, DEVICE)

    unet_scheduler.step()
    current_lr = unet_scheduler.get_last_lr()[0]

    unet_history["train_loss"].append(train_loss)
    unet_history["train_ce"].append(train_ce)
    unet_history["train_dice"].append(train_dice)
    unet_history["val_loss"].append(val_loss)
    unet_history["val_miou"].append(val_miou)
    unet_history["val_dice"].append(val_dice)
    unet_history["lr"].append(current_lr)

    ep_time = time.time() - ep_start
    print(f"Epoch [{epoch:02d}/{SEG_EPOCHS}] "
          f"TrainLoss={train_loss:.4f} "
          f"ValLoss={val_loss:.4f} "
          f"ValmIoU={val_miou:.4f} "
          f"ValDice={val_dice:.4f} "
          f"LR={current_lr:.2e} "
          f"Time={ep_time:.1f}s")

    # Save best model
    if val_miou > best_unet_miou:
        best_unet_miou  = val_miou
        best_unet_epoch = epoch
        unet_patience   = 0
        model_to_save   = unet.module if hasattr(unet, "module") else unet
        torch.save(model_to_save.state_dict(),
                   str(CKPT_DIR / "unet_best.pt"))
        print(f"  ✅ Best U-Net saved (mIoU={best_unet_miou:.4f})")
    else:
        unet_patience += 1
        if unet_patience >= EARLY_STOP:
            print(f"\nEarly stopping at epoch {epoch}. "
                  f"Best mIoU={best_unet_miou:.4f} at epoch {best_unet_epoch}")
            break

unet_train_time = (time.time() - unet_train_start) / 60
print(f"\nU-Net training complete in {unet_train_time:.1f} minutes")
print(f"Best Val mIoU : {best_unet_miou:.4f} at epoch {best_unet_epoch}")

In [ ]:
#3.3.4
epochs_ran = len(unet_history["train_loss"])
ep_range   = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("U-Net Training Curves", fontsize=14, fontweight="bold")

axes[0].plot(ep_range, unet_history["train_loss"],
             label="Train Loss", color="royalblue", linewidth=2)
axes[0].plot(ep_range, unet_history["val_loss"],
             label="Val Loss",   color="tomato",    linewidth=2)
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_range, unet_history["val_miou"],
             color="green", linewidth=2)
axes[1].set_title("Validation mIoU")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("mIoU")
axes[1].grid(True, alpha=0.3)

axes[2].plot(ep_range, unet_history["val_dice"],
             color="purple", linewidth=2)
axes[2].set_title("Validation Dice Score")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Dice")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "unet_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: unet_training_curves.png")

pd.DataFrame(unet_history).to_csv(
    RESULTS_DIR / "unet_training_history.csv", index=False)
print("Saved: unet_training_history.csv")

In [ ]:
#3.3.5
# Load best U-Net weights
unet_eval = UNet(in_channels=3, num_classes=NUM_SEG_CLASSES).to(DEVICE)
unet_eval.load_state_dict(torch.load(str(CKPT_DIR / "unet_best.pt"),
                                      map_location=DEVICE))
unet_eval.eval()

per_class_iou_unet  = np.zeros(NUM_SEG_CLASSES)
per_class_dice_unet = np.zeros(NUM_SEG_CLASSES)
class_counts_unet   = np.zeros(NUM_SEG_CLASSES)

total_miou_unet = 0.0
total_dice_unet = 0.0
n_batches       = 0
unet_infer_times = []

with torch.no_grad():
    for imgs, masks in tqdm(test_loader, desc="Evaluating U-Net"):
        imgs  = imgs.to(DEVICE)
        masks = masks.to(DEVICE)

        t0     = time.time()
        logits = unet_eval(imgs)
        t1     = time.time()
        unet_infer_times.append((t1 - t0) / imgs.shape[0] * 1000)

        pred_labels = logits.argmax(dim=1).cpu()
        masks_cpu   = masks.cpu()

        for cls in range(NUM_SEG_CLASSES):
            pred_c   = (pred_labels == cls)
            target_c = (masks_cpu   == cls)
            valid    = (masks_cpu   != IGNORE_INDEX)
            pred_c   = pred_c & valid
            target_c = target_c & valid
            inter    = (pred_c & target_c).sum().item()
            union    = (pred_c | target_c).sum().item()
            if union > 0:
                per_class_iou_unet[cls]  += inter / union
                class_counts_unet[cls]   += 1
            tp    = inter
            fp_fn = pred_c.sum().item() + target_c.sum().item()
            if fp_fn > 0:
                per_class_dice_unet[cls] += 2 * tp / fp_fn

        total_miou_unet += compute_miou(logits.cpu(), masks_cpu)
        total_dice_unet += compute_dice(logits.cpu(), masks_cpu)
        n_batches       += 1

per_class_iou_unet  = per_class_iou_unet  / np.maximum(class_counts_unet, 1)
per_class_dice_unet = per_class_dice_unet / np.maximum(class_counts_unet, 1)
mean_miou_unet      = total_miou_unet / n_batches
mean_dice_unet      = total_dice_unet / n_batches
avg_infer_ms_unet   = np.mean(unet_infer_times)

print("=== U-Net Test Set Evaluation ===")
print(f"Mean mIoU        : {mean_miou_unet:.4f}")
print(f"Mean Dice        : {mean_dice_unet:.4f}")
print(f"Inference speed  : {avg_infer_ms_unet:.2f} ms/image")
print(f"\nPer-class IoU:")
for i, cls in enumerate(SEG_CLASSES):
    print(f"  {cls:<15}: IoU={per_class_iou_unet[i]:.4f}  "
          f"Dice={per_class_dice_unet[i]:.4f}")

df_unet_results = pd.DataFrame({
    "class" : SEG_CLASSES + ["MEAN"],
    "iou"   : list(per_class_iou_unet) + [mean_miou_unet],
    "dice"  : list(per_class_dice_unet) + [mean_dice_unet],
    "model" : "unet",
})
df_unet_results.to_csv(RESULTS_DIR / "unet_results.csv", index=False)
print("\nSaved: unet_results.csv")

unet_timing = {
    "model"              : "unet",
    "train_time_minutes" : round(unet_train_time, 2),
    "inference_ms"       : round(avg_infer_ms_unet, 2),
    "mean_miou"          : round(mean_miou_unet, 4),
    "mean_dice"          : round(mean_dice_unet, 4),
}
pd.DataFrame([unet_timing]).to_csv(RESULTS_DIR / "unet_timing.csv", index=False)
print("Saved: unet_timing.csv")

In [ ]:
#3.3.6
def visualize_unet_predictions(model, dataset, n=8, save_path=None):
    model.eval()
    indices = np.random.choice(len(dataset), size=n, replace=False)

    fig, axes = plt.subplots(n, 3, figsize=(15, n * 4))
    fig.suptitle("U-Net Predictions — Input | Ground Truth | Prediction",
                 fontsize=13, fontweight="bold")

    col_titles = ["Input Image", "Ground Truth", "U-Net Prediction"]
    for ax, title in zip(axes[0], col_titles):
        ax.set_title(title, fontsize=11, fontweight="bold")

    with torch.no_grad():
        for row, idx in enumerate(indices):
            img_t, msk_t = dataset[idx]
            img_input    = img_t.unsqueeze(0).to(DEVICE)

            logits = model(img_input)
            pred   = logits.argmax(dim=1).squeeze(0).cpu().numpy()

            img_vis = denormalize(img_t).permute(1, 2, 0).numpy()
            gt_vis  = mask_to_color(msk_t.numpy())
            pr_vis  = mask_to_color(pred)

            axes[row][0].imshow(img_vis)
            axes[row][0].axis("off")
            axes[row][1].imshow(gt_vis)
            axes[row][1].axis("off")
            axes[row][2].imshow(pr_vis)
            axes[row][2].axis("off")

    handles = [patches.Patch(color=plt.cm.tab10(i / NUM_SEG_CLASSES),
                              label=SEG_CLASSES[i])
               for i in range(NUM_SEG_CLASSES)]
    fig.legend(handles=handles, loc="lower center", ncol=5,
               fontsize=8, bbox_to_anchor=(0.5, -0.01))

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


visualize_unet_predictions(
    unet_eval,
    test_dataset,
    n         = 8,
    save_path = RESULTS_DIR / "unet_qualitative.png"
)

In [ ]:
#3.3.7
if torch.cuda.is_available():
    print("=== GPU Memory Usage (U-Net) ===")
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1e6
        reserved  = torch.cuda.memory_reserved(i) / 1e6
        total     = torch.cuda.get_device_properties(i).total_memory / 1e6
        print(f"  GPU {i}: Allocated={allocated:.1f}MB  "
              f"Reserved={reserved:.1f}MB  Total={total:.1f}MB")

**Evaluation** 

In [ ]:
#3.4.1
np.random.seed(SEED)
indices = np.random.choice(len(test_dataset), size=8, replace=False)

fig, axes = plt.subplots(8, 4, figsize=(20, 8 * 4))
fig.suptitle("Segmentation Comparison — Input | Ground Truth | SegFormer | U-Net",
             fontsize=14, fontweight="bold")

col_titles = ["Input Image", "Ground Truth", "SegFormer", "U-Net"]
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=11, fontweight="bold")

model_to_eval.eval()
unet_eval.eval()

with torch.no_grad():
    for row, idx in enumerate(indices):
        img_t, msk_t = test_dataset[idx]
        img_input    = img_t.unsqueeze(0).to(DEVICE)

        # SegFormer prediction
        seg_out    = model_to_eval(pixel_values=img_input)
        seg_logits = torch.nn.functional.interpolate(
            seg_out.logits,
            size=msk_t.shape[-2:],
            mode="bilinear", align_corners=False,
        )
        seg_pred = seg_logits.argmax(dim=1).squeeze(0).cpu().numpy()

        # U-Net prediction
        unet_logits = unet_eval(img_input)
        unet_pred   = unet_logits.argmax(dim=1).squeeze(0).cpu().numpy()

        img_vis = denormalize(img_t).permute(1, 2, 0).numpy()
        gt_vis  = mask_to_color(msk_t.numpy())
        seg_vis = mask_to_color(seg_pred)
        une_vis = mask_to_color(unet_pred)

        axes[row][0].imshow(img_vis);  axes[row][0].axis("off")
        axes[row][1].imshow(gt_vis);   axes[row][1].axis("off")
        axes[row][2].imshow(seg_vis);  axes[row][2].axis("off")
        axes[row][3].imshow(une_vis);  axes[row][3].axis("off")

handles = [patches.Patch(color=plt.cm.tab10(i / NUM_SEG_CLASSES),
                          label=SEG_CLASSES[i])
           for i in range(NUM_SEG_CLASSES)]
fig.legend(handles=handles, loc="lower center", ncol=5,
           fontsize=9, bbox_to_anchor=(0.5, -0.01))

plt.tight_layout()
plt.savefig(RESULTS_DIR / "seg_side_by_side_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: seg_side_by_side_comparison.png")

In [ ]:
#3.4.2
x      = np.arange(NUM_SEG_CLASSES)
width  = 0.35

fig, ax = plt.subplots(figsize=(16, 6))
fig.suptitle("Per-Class IoU — SegFormer vs U-Net", fontsize=14, fontweight="bold")

bars1 = ax.bar(x - width/2, per_class_iou,      width, label="SegFormer",
               color="royalblue", edgecolor="black")
bars2 = ax.bar(x + width/2, per_class_iou_unet, width, label="U-Net",
               color="tomato",    edgecolor="black")

ax.set_xticks(x)
ax.set_xticklabels(SEG_CLASSES, rotation=45, ha="right")
ax.set_ylabel("IoU Score")
ax.set_ylim(0, 1.0)
ax.legend(fontsize=11)
ax.axhline(mean_miou,      color="royalblue", linestyle="--",
           linewidth=1.5, alpha=0.7, label=f"SegFormer Mean={mean_miou:.3f}")
ax.axhline(mean_miou_unet, color="tomato",    linestyle="--",
           linewidth=1.5, alpha=0.7, label=f"U-Net Mean={mean_miou_unet:.3f}")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis="y")

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", fontsize=7, rotation=45)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", fontsize=7, rotation=45)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "seg_per_class_iou_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: seg_per_class_iou_comparison.png")

In [ ]:
#3.4.3
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Training Curves — SegFormer vs U-Net", fontsize=14, fontweight="bold")

seg_ep  = range(1, len(seg_history["val_loss"]) + 1)
unet_ep = range(1, len(unet_history["val_loss"]) + 1)

# Loss
axes[0].plot(seg_ep,  seg_history["train_loss"],  color="royalblue",
             linewidth=2, label="SegFormer Train")
axes[0].plot(seg_ep,  seg_history["val_loss"],    color="royalblue",
             linewidth=2, linestyle="--", label="SegFormer Val")
axes[0].plot(unet_ep, unet_history["train_loss"], color="tomato",
             linewidth=2, label="U-Net Train")
axes[0].plot(unet_ep, unet_history["val_loss"],   color="tomato",
             linewidth=2, linestyle="--", label="U-Net Val")
axes[0].set_title("Loss Curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# mIoU
axes[1].plot(seg_ep,  seg_history["val_miou"],  color="royalblue",
             linewidth=2, label="SegFormer")
axes[1].plot(unet_ep, unet_history["val_miou"], color="tomato",
             linewidth=2, label="U-Net")
axes[1].set_title("Validation mIoU")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("mIoU")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Dice
axes[2].plot(seg_ep,  seg_history["val_dice"],  color="royalblue",
             linewidth=2, label="SegFormer")
axes[2].plot(unet_ep, unet_history["val_dice"], color="tomato",
             linewidth=2, label="U-Net")
axes[2].set_title("Validation Dice Score")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Dice")
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "seg_combined_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: seg_combined_training_curves.png")

In [ ]:
#3.4.4
# Load saved CSVs in case session restarted
df_seg_res  = pd.read_csv(RESULTS_DIR / "segformer_results.csv")
df_unet_res = pd.read_csv(RESULTS_DIR / "unet_results.csv")
df_seg_time = pd.read_csv(RESULTS_DIR / "segformer_timing.csv")
df_unet_time= pd.read_csv(RESULTS_DIR / "unet_timing.csv")

# Per-class comparison table
df_comparison = pd.merge(
    df_seg_res.rename(columns={"iou": "SegFormer_IoU", "dice": "SegFormer_Dice"}),
    df_unet_res.rename(columns={"iou": "UNet_IoU",     "dice": "UNet_Dice"}),
    on="class"
).drop(columns=["model_x", "model_y"])

df_comparison["IoU_Delta"] = (
    df_comparison["SegFormer_IoU"] - df_comparison["UNet_IoU"]
).round(4)

print("=== Per-Class Segmentation Comparison ===\n")
print(df_comparison.to_string(index=False))

# Summary comparison
print("\n=== Model Summary Comparison ===")
summary = pd.DataFrame([
    {
        "Model"             : "SegFormer-b0",
        "Mean mIoU"         : df_seg_time["mean_miou"].values[0],
        "Mean Dice"         : df_seg_time["mean_dice"].values[0],
        "Train Time (min)"  : df_seg_time["train_time_minutes"].values[0],
        "Inference (ms/img)": df_seg_time["inference_ms"].values[0],
    },
    {
        "Model"             : "U-Net (scratch)",
        "Mean mIoU"         : df_unet_time["mean_miou"].values[0],
        "Mean Dice"         : df_unet_time["mean_dice"].values[0],
        "Train Time (min)"  : df_unet_time["train_time_minutes"].values[0],
        "Inference (ms/img)": df_unet_time["inference_ms"].values[0],
    },
])
print(summary.to_string(index=False))

# Save final segmentation_results.csv (required deliverable)
df_comparison.to_csv(RESULTS_DIR / "segmentation_results.csv", index=False)
summary.to_csv(RESULTS_DIR / "segmentation_summary.csv", index=False)
print("\nSaved: segmentation_results.csv")
print("Saved: segmentation_summary.csv")

In [ ]:
#3.4.5
def find_failure_cases(model, dataset, model_name, n=5,
                       is_segformer=False, save_path=None):
    """Find worst predictions by lowest per-image mIoU."""
    model.eval()
    scores = []

    with torch.no_grad():
        for idx in range(len(dataset)):
            img_t, msk_t = dataset[idx]
            img_input    = img_t.unsqueeze(0).to(DEVICE)

            if is_segformer:
                out    = model(pixel_values=img_input)
                logits = torch.nn.functional.interpolate(
                    out.logits, size=msk_t.shape[-2:],
                    mode="bilinear", align_corners=False)
            else:
                logits = model(img_input)

            miou = compute_miou(logits.cpu(), msk_t.unsqueeze(0))
            scores.append((miou, idx))

    # Sort by lowest mIoU — these are failure cases
    scores.sort(key=lambda x: x[0])
    worst_indices = [idx for _, idx in scores[:n]]

    fig, axes = plt.subplots(n, 3, figsize=(15, n * 4))
    fig.suptitle(f"Failure Cases — {model_name}\n(Lowest mIoU predictions)",
                 fontsize=13, fontweight="bold")

    col_titles = ["Input Image", "Ground Truth", f"{model_name} Prediction"]
    for ax, title in zip(axes[0], col_titles):
        ax.set_title(title, fontsize=10, fontweight="bold")

    with torch.no_grad():
        for row, idx in enumerate(worst_indices):
            img_t, msk_t = dataset[idx]
            img_input    = img_t.unsqueeze(0).to(DEVICE)

            if is_segformer:
                out    = model(pixel_values=img_input)
                logits = torch.nn.functional.interpolate(
                    out.logits, size=msk_t.shape[-2:],
                    mode="bilinear", align_corners=False)
            else:
                logits = model(img_input)

            pred    = logits.argmax(dim=1).squeeze(0).cpu().numpy()
            miou    = compute_miou(logits.cpu(), msk_t.unsqueeze(0))

            img_vis = denormalize(img_t).permute(1, 2, 0).numpy()
            gt_vis  = mask_to_color(msk_t.numpy())
            pr_vis  = mask_to_color(pred)

            axes[row][0].imshow(img_vis)
            axes[row][0].set_ylabel(f"mIoU={miou:.3f}", fontsize=9)
            axes[row][0].axis("off")
            axes[row][1].imshow(gt_vis);  axes[row][1].axis("off")
            axes[row][2].imshow(pr_vis);  axes[row][2].axis("off")

    handles = [patches.Patch(color=plt.cm.tab10(i / NUM_SEG_CLASSES),
                              label=SEG_CLASSES[i])
               for i in range(NUM_SEG_CLASSES)]
    fig.legend(handles=handles, loc="lower center", ncol=5,
               fontsize=8, bbox_to_anchor=(0.5, -0.01))

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


find_failure_cases(
    model_to_eval, test_dataset,
    model_name   = "SegFormer",
    n            = 5,
    is_segformer = True,
    save_path    = RESULTS_DIR / "segformer_failure_cases.png"
)

find_failure_cases(
    unet_eval, test_dataset,
    model_name   = "U-Net",
    n            = 5,
    is_segformer = False,
    save_path    = RESULTS_DIR / "unet_failure_cases.png"
)